# CXF (Control eXchange Format) Ingestion

[CXF](https://github.com/NREL/ctrl-flow-framework)/S231 is the JSON-LD serialization the
LBNL Modelica Buildings library uses to publish its ASHRAE Guideline 36 control
sequences (`Buildings.Controls.OBC.ASHRAE.G36.*`) - vendored here under
`src/semantic_objects/ontologies/cxf/` as ~45 files, one per control block (a
`Controller`, or a smaller `Subsequences.*` building block it composes).

This is **not** the same thing as the `g36:` extension already living inside
`223p.ttl` (see `tutorial/g36-extension-tutorial.ipynb`) - that `g36:` namespace
models *physical equipment classes* (a `Zone` with a CO2 point, a `Fan` with a
start/stop command) as S223 SHACL shapes. CXF instead models *control-sequence
blocks* - Modelica-style named inputs/outputs/parameters, roughly a function
signature - with no SHACL, no `rdfs:subClassOf` hierarchy, and no relation to the
S223 RDF graph at all. The two happen to share the string "G36" because they both
implement the same ASHRAE guideline, from two unrelated standards efforts.

Because the source shape is so different, CXF gets its own parser/emitter
(`semantic_objects.ingest.cxf`) rather than another `OntologyAdapter` plugged into
the SHACL walker used for s223/g36/watr - see `ingest/cxf/parser.py`'s docstring.
Regenerate with `python -m semantic_objects.ingest.cli --ontology cxf`.

This tutorial: (1) surveys everything the ingest pulled in, then (2) looks closely
at two specific blocks.

## 1. Survey: everything the ingest imported

`semantic_objects.cxf.blocks` is a flat re-export of all 37 generated block
classes (mirroring the CXF folder hierarchy underneath, in `_generated/blocks/**`,
one module per block); `semantic_objects.cxf.enumerationkinds` is the 10 CXF
`Types/*.jsonld` enumerations.

In [1]:
import semantic_objects.cxf as cxf

print(f"{len(cxf.blocks.__all__)} blocks, {len(cxf.enumerationkinds.__all__)} enumeration-kind classes")
print()
print("blocks:", sorted(cxf.blocks.__all__))

CRITICAL:root:Install the 'bacnet-ingress' module, e.g. 'pip install buildingmotif[bacnet-ingress]'


37 blocks, 71 enumeration-kind classes

blocks: ['AirEconomizerHighLimits', 'Ashrae621AHU', 'Ashrae621Setpoints', 'Common', 'ControlLoops', 'CoolingOnlyController', 'CoolingOnlySubsequencesActiveAirFlow', 'CoolingOnlySubsequencesAlarms', 'CoolingOnlySubsequencesSystemRequests', 'DamperValves', 'Dampers', 'EconomizersController', 'Enable', 'FreezeProtection', 'Overrides', 'PlantRequests', 'ReheatController', 'ReheatSubsequencesActiveAirFlow', 'ReheatSubsequencesAlarms', 'ReheatSubsequencesSystemRequests', 'ReliefDamper', 'ReliefFan', 'Reliefs', 'ReturnFan', 'ReturnFanAirflowTracking', 'ReturnFanDirectPressure', 'SeparateWithAFMS', 'SeparateWithDP', 'SupplyFan', 'SupplySignals', 'SupplyTemperature', 'TimeSuppression', 'Title24AHU', 'Title24Setpoints', 'TrimAndRespond', 'VavController', 'ZoneStates']


Three different source files each define a block whose *own* name is `Controller`
(one per equipment family - AHU, cooling-only terminal unit, reheat terminal
unit), and two different `Subsequences.Alarms`/`ActiveAirFlow`/`SystemRequests`
files exist under sibling `CoolingOnly`/`Reheat` folders. Flattening all 37 into
one importable namespace means those collisions have to be resolved somehow - the
emitter aliases every colliding name using just enough of its source path to stay
unique (see `_disambiguating_aliases` in `ingest/cxf/emitter.py`), rather than
letting the second import silently shadow the first:

In [2]:
import re

for name in sorted(cxf.blocks.__all__):
    if re.search(r'(Controller|Alarms|ActiveAirFlow|SystemRequests|Setpoints|AHU)$', name):
        cls = getattr(cxf.blocks, name)
        print(f"{name:28s} <- {cls._name}")

Ashrae621AHU                 <- Buildings.Controls.OBC.ASHRAE.G36.AHUs.MultiZone.VAV.SetPoints.OutdoorAirFlow.ASHRAE62_1.AHU
Ashrae621Setpoints           <- Buildings.Controls.OBC.ASHRAE.G36.VentilationZones.ASHRAE62_1.Setpoints
CoolingOnlyController        <- Buildings.Controls.OBC.ASHRAE.G36.TerminalUnits.CoolingOnly.Controller
CoolingOnlySubsequencesActiveAirFlow <- Buildings.Controls.OBC.ASHRAE.G36.TerminalUnits.CoolingOnly.Subsequences.ActiveAirFlow
CoolingOnlySubsequencesAlarms <- Buildings.Controls.OBC.ASHRAE.G36.TerminalUnits.CoolingOnly.Subsequences.Alarms
CoolingOnlySubsequencesSystemRequests <- Buildings.Controls.OBC.ASHRAE.G36.TerminalUnits.CoolingOnly.Subsequences.SystemRequests
EconomizersController        <- Buildings.Controls.OBC.ASHRAE.G36.AHUs.MultiZone.VAV.Economizers.Controller
ReheatController             <- Buildings.Controls.OBC.ASHRAE.G36.TerminalUnits.Reheat.Controller
ReheatSubsequencesActiveAirFlow <- Buildings.Controls.OBC.ASHRAE.G36.TerminalUnits.Reheat.Sub

Grouping by top-level CXF category (the first folder under `.../G36/`) shows
where the 37 blocks come from - mostly terminal-unit and AHU control sequences,
plus the shared `Generic` building blocks (trim-and-respond, time suppression,
economizer high-limit curves) every equipment-specific controller composes:

In [3]:
from collections import Counter

category_counts = Counter(cls._name.split('.')[5] for cls in (getattr(cxf.blocks, n) for n in cxf.blocks.__all__))
for category, count in sorted(category_counts.items(), key=lambda kv: -kv[1]):
    print(f"{category:20s} {count}")

AHUs                 19
TerminalUnits        11
Generic              3
VentilationZones     2
ThermalZones         2


### Enumeration kinds (`Types/*.jsonld`)

10 enumeration types, each with a handful of named literal values - these back
the `EnumParameter` fields seen below (e.g. `venStd: VentilationStandard =
ASHRAE62_1`). Two different climate-zone enumerations (`ASHRAEClimateZone`,
`Title24ClimateZone`) share several literal names (`Not_Specified`, `Zone_7`,
`Zone_8`), so those literals are aliased the same way the blocks above are - e.g.
`ASHRAEClimateZone_Not_Specified` rather than a bare `Not_Specified` that would
collide with `Title24ClimateZone`'s own literal of the same name.

In [4]:
from semantic_objects.cxf.core import EnumerationKind as CxfEnumerationKind

# An enum *type* is generated as `class X(EnumerationKind)`; its *literals* are
# `class Lit(X)` - the immediate base class is the structural signal, not the
# name (several literals keep their bare source name, e.g. 'ASHRAE62_1', so a
# name-pattern heuristic can't reliably tell type names from literal names).
type_names = sorted(n for n in cxf.enumerationkinds.__all__
                     if getattr(cxf.enumerationkinds, n).__bases__[0] is CxfEnumerationKind)

for type_name in type_names:
    cls = getattr(cxf.enumerationkinds, type_name)
    literals = [n for n in cxf.enumerationkinds.__all__
                if getattr(cxf.enumerationkinds, n).__bases__[0] is cls]
    print(f"{type_name} ({cls.comment})")
    for lit in literals:
        print(f"    {lit}")
    print()

ASHRAEClimateZone (Enumeration of ASHRAE climate zone)
    ASHRAEClimateZone_Not_Specified
    Zone_1A
    Zone_1B
    Zone_2A
    Zone_2B
    Zone_3A
    Zone_3B
    Zone_3C
    Zone_4A
    Zone_4B
    Zone_4C
    Zone_5A
    Zone_5B
    Zone_5C
    Zone_6A
    Zone_6B
    ASHRAEClimateZone_Zone_7
    ASHRAEClimateZone_Zone_8

ControlEconomizer (Enumeration to configure the economizer enable and disable control)
    DifferentialDryBulb
    DifferentialEnthalpyWithFixedDryBulb
    FixedDryBulb
    FixedDryBulbWithDifferentialDryBulb
    FixedEnthalpyWithFixedDryBulb

CoolingCoil (Enumeration to configure the cooling coil)
    DXCoil
    CoolingCoil_None_
    CoolingCoil_WaterBased

EnergyStandard (Enumeration to configure the energy standard)
    ASHRAE90_1
    EnergyStandard_California_Title_24

FreezeStat (Enumeration of different freeze stat options)
    Hardwired_to_BAS
    Hardwired_to_equipment
    No_freeze_stat

HeatingCoil (Enumeration to configure the heating coil)
    Electr

## 2. Close-up #1: `TerminalUnits.CoolingOnly.Controller`

The top-level controller for a cooling-only VAV terminal box (ASHRAE Guideline
36 §5.5) - a good first example because it has a realistic mix of everything:
real-valued I/O with units, boolean/integer signals without units, an enum
parameter, an untyped parameter the source data just doesn't specify, and nine
composed sub-blocks.

In [5]:
Controller = cxf.blocks.CoolingOnlyController
print(Controller.comment[:400], "...")

info=<html>
<p>
Controller for cooling only terminal box according to Section 5.5 of ASHRAE
Guideline 36, May 2020. It outputs discharge airflow setpoint <code>VSet_flow</code>,
damper position setpoint <code>yDam</code>, AHU cooling supply temperature
setpoint reset request <code>yZonTemResReq</code>, and static pressure setpoint
reset request <code>yZonPreResReq</code>. It also outputs the alarm ...


**Inputs and outputs** - every `Real` one carries a `qk`/`unit` pair; `Boolean`/`Integer` ones don't (CXF has no unit for a status flag or a mode index):

In [6]:
inst = Controller()

def port_kind(field):
    return field.metadata.get('kind')

for direction in ('input', 'output'):
    print(f"--- {direction}s ---")
    for name, f in Controller.__dataclass_fields__.items():
        if port_kind(f) != direction:
            continue
        port = getattr(inst, name)
        qk = getattr(port, 'qk', None)
        unit = getattr(port, 'unit', None)
        qk, unit = (qk.__name__ if qk else '-'), (unit.__name__ if unit else '-')
        print(f"  {name:16s} {qk:24s} {unit:12s} {port.description}")
    print()

--- inputs ---
  oveDamPos        -                        -            Index of overriding damper position, 1: set to close; 2: set to open
  oveFloSet        -                        -            Index of overriding flow setpoint, 1: set to zero; 2: set to cooling maximum; 3: set to minimum flow; 4: set to heating maximum
  ppmCO2           -                        -            Detected CO2 concentration
  ppmCO2Set        -                        -            CO2 concentration setpoint
  TCooSet          ThermodynamicTemperature K            Zone cooling setpoint temperature
  TDis             ThermodynamicTemperature K            Measured discharge air temperature
  THeaSet          ThermodynamicTemperature K            Zone heating setpoint temperature
  TSup             ThermodynamicTemperature K            Temperature of the air supplied from central air handler
  TZon             ThermodynamicTemperature K            Measured room temperature
  u1Fan            -               

**Parameters** - same shape, plus a default value pulled straight from the source (`S231:value`):

In [7]:
for name, f in Controller.__dataclass_fields__.items():
    if port_kind(f) != 'parameter':
        continue
    p = getattr(inst, name)
    qk = getattr(p, 'qk', None)
    unit = getattr(p, 'unit', None)
    print(f"{name:20s} default={p.value!r:>10}  qk={qk.__name__ if qk else '-':22s} unit={unit.__name__ if unit else '-'}")

chaRat               default=       540  qk=-                      unit=K
damCon               default='Buildings.Controls.OBC.CDL.Types.SimpleController.PI'  qk=-                      unit=-
damPosHys            default=     0.005  qk=-                      unit=UNITLESS
dTHys                default=      0.25  qk=-                      unit=K
durTimFlo            default=        60  qk=-                      unit=SEC
durTimTem            default=       120  qk=-                      unit=SEC
fanOffTim            default=       600  qk=-                      unit=SEC
floHys               default='0.01*VMin_flow'  qk=-                      unit=M3_PER_SEC
have_CO2Sen          default=      True  qk=-                      unit=-
have_occSen          default=      True  qk=-                      unit=-
have_winSen          default=      True  qk=-                      unit=-
iniDam               default=      0.01  qk=-                      unit=UNITLESS
kCooCon              default=    

`venStd` has no default and no unit/quantitykind at all - its source JSON-LD node
has *no `@type` whatsoever* (not even `S231:Parameter`), just a label and
description. Rather than silently dropping it, the parser still captures the
name/description and the emitter falls back to a bare `Parameter` (see
`ingest/cxf/core.py::Parameter`'s docstring). `damCon`'s default names a
`Buildings.Controls.OBC.CDL.*` type - CDL is a different Modelica library this
corpus doesn't vendor, so it's kept as a descriptive string rather than invented
as a fake enumeration:

In [8]:
print("venStd:", inst.venStd)
print("damCon:", inst.damCon)

venStd: Parameter(description='Ventilation standard, ASHRAE 62.1 or Title 24', value=None)
damCon: Parameter(description='Type of controller', value='Buildings.Controls.OBC.CDL.Types.SimpleController.PI')


Compare that to `AHUs.MultiZone.VAV.Controller`, whose `ashCliZon` parameter
*also* has no `S231:isOfDataType` - but its default value is the fully-dotted
literal path `...Types.ASHRAEClimateZone.Not_Specified`. The parser recovers the
real enum type from that dotted default (`ingest/cxf/parser.py::
_resolve_unknown_enum_defaults`) instead of leaving it 'Unknown' like `venStd`:

In [9]:
vav = cxf.blocks.VavController()
print("ashCliZon.enumeration_kind:", vav.ashCliZon.enumeration_kind)
print("ashCliZon.value:", vav.ashCliZon.value, "-", vav.ashCliZon.value.comment)

ashCliZon.enumeration_kind: <class 'semantic_objects.cxf._generated.enumerationkinds.ASHRAEClimateZone'>
ashCliZon.value: <class 'semantic_objects.cxf._generated.enumerationkinds.ASHRAEClimateZone_Not_Specified'> - Not specified


**Sub-block composition** - `Controller` is built from nine named sub-blocks;
each is recorded as a `SubBlock(block_type=..., description=...)`, i.e. *which*
generated class fills that role, not a live instance wired up to it (CXF's
`isConnectedTo` port wiring is out of scope here - see the intro). Two of them,
`minFlo` and `setPoi`, both instantiate a block whose own name is `Setpoints` -
one from `VentilationZones/Title24/`, the other from `VentilationZones/ASHRAE62_1/` -
so they resolve to two genuinely different classes despite sharing a bare name:

In [10]:
for name, f in Controller.__dataclass_fields__.items():
    if port_kind(f) != 'subblock':
        continue
    sub = getattr(inst, name)
    print(f"{name:10s} -> {sub.block_type.__name__ if sub.block_type else '(external)':20s} {sub.description}")

print()
minFlo_type, setPoi_type = inst.minFlo.block_type, inst.setPoi.block_type
print("minFlo and setPoi are both named 'Setpoints', but distinct classes:", minFlo_type is not setPoi_type)
print(" ", minFlo_type.__module__, "->", minFlo_type._name)
print(" ", setPoi_type.__module__, "->", setPoi_type._name)

actAirSet  -> ActiveAirFlow        Active airflow setpoint
ala        -> Alarms               Generate alarms
conLoo     -> ControlLoops         Heating and cooling control loop
dam        -> Dampers              Damper control
minFlo     -> Setpoints            Output the minimum outdoor airflow rate setpoint, when using Title 24
setPoi     -> Setpoints            Output the minimum outdoor airflow rate setpoint, when using ASHRAE 62.1
sysReq     -> SystemRequests       Specify system requests 
timSup     -> TimeSuppression      Specify suppresion time due to the setpoint change and check if it has passed the suppresion period
zonSta     -> ZoneStates           Check if the zone is in cooling state

minFlo and setPoi are both named 'Setpoints', but distinct classes: True
  semantic_objects.cxf._generated.blocks.ventilation_zones.title24.setpoints -> Buildings.Controls.OBC.ASHRAE.G36.VentilationZones.Title24.Setpoints
  semantic_objects.cxf._generated.blocks.ventilation_zones.ashrae62_

## 3. Close-up #2: `Generic.TrimAndRespond`

The trim-and-respond logic (Guideline 36 §5.1.14.3/4) used by most of the
reset-request sequences above (`sysReq` on the cooling-only Controller, plant
requests, static pressure reset, ...). A good second example for the opposite
reason from `Controller`: a *small, clean* I/O signature (4 ports, 10
parameters) wrapped around a much larger pile of internal CDL primitive blocks
(comparators, switches, timers) that fall outside the ~45-file G36 corpus this
ingest vendors - a real look at how unresolvable sub-blocks are handled rather
than silently dropped.

In [11]:
TrimAndRespond = cxf.blocks.TrimAndRespond
tr = TrimAndRespond()
print(TrimAndRespond.comment[:300], "...")
print()
for name, f in TrimAndRespond.__dataclass_fields__.items():
    kind = port_kind(f)
    if kind in ('input', 'output'):
        port = getattr(tr, name)
        print(f"{kind:9s} {name:12s} {port.description}")
    elif kind == 'parameter':
        p = getattr(tr, name)
        print(f"{kind:9s} {name:12s} default={p.value!r} {getattr(p, 'unit', None) and p.unit.__name__ or ''}")

info=<html>
<p>
This block implements the trim and respond logic according to Section 5.1.14.3 
and 5.1.14.4 of ASHRAE Guideline 36, May 2020.
</p>
<p>
For each upstream system or plant set point being controlled by a trim and respond
loop, define the initial values in system or plant sequences. Val ...

input     numOfReq     Number of requests from zones/systems
input     uDevSta      On/Off status of the associated device
input     uHol         Hold signal
output    y            Setpoint that have been reset
parameter delTim       default=None SEC
parameter dtHol        default=0 SEC
parameter have_hol     default=False 
parameter iniSet       default=None 
parameter maxRes       default=None 
parameter maxSet       default=None 
parameter minSet       default=None 
parameter numIgnReq    default=None 
parameter resAmo       default=None 
parameter samplePeriod default=None SEC
parameter triAmo       default=None 


In [12]:
external_sub_blocks = [
    (name, getattr(tr, name)) for name, f in TrimAndRespond.__dataclass_fields__.items()
    if port_kind(f) == 'subblock' and getattr(tr, name).block_type is None
]
print(f"{len(external_sub_blocks)} of TrimAndRespond's sub-blocks are CDL primitives outside this corpus, e.g.:")
for name, sub in external_sub_blocks[:6]:
    print(f"  {name:12s} {sub.description}")

45 of TrimAndRespond's sub-blocks are CDL primitives outside this corpus, e.g.:
  abs          [external type: Buildings.Controls.OBC.CDL.Reals.Abs] Absolute value of real input
  abs1         [external type: Buildings.Controls.OBC.CDL.Reals.Abs] Absolute value of real input
  add1         [external type: Buildings.Controls.OBC.CDL.Reals.Add] Increase setpoint by amount of value defined from reset logic
  add2         [external type: Buildings.Controls.OBC.CDL.Reals.Add] Net reset value
  and2         [external type: Buildings.Controls.OBC.CDL.Logical.And] After (device ON + delTim + timSta), when request number becomes more than ignored requests number
  assMes       [external type: Buildings.Controls.OBC.CDL.Utilities.Assert] Generate alarm message


## Scope recap

This ingest captures, per block: **inputs, outputs, parameters, their
descriptions, quantitykinds, and units** - plus, one level further than the
original brief, *which named sub-block* composes a block (structural
composition). It deliberately does **not** capture the wiring between those
sub-blocks' ports (`S231:isConnectedTo`) or pinned-to-a-sibling-parameter values
(`S231:isFinal`) - reconstructing the full executable control diagram was never
the goal, just the block-level contract a modeler would need to know a sequence
exists and what it needs/produces.